# Results & Model Comparison

This notebook compares the Log-MLR baseline and Bingham physics model
using RMSE, and visualizes model predictions against observed data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.optimize import curve_fit


In [ ]:
# Load dataset (skip the units row)
file_path = "../data/USROP_A 3 N-SH-F-15d - Final.csv"
df = pd.read_csv(file_path, skiprows=[1])
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## Fit Models

Re-fit both models so this notebook is self-contained.


In [ ]:
# Filter non-physical rows
df_bingham = df[(df["ROP"] > 0) & (df["WOB"] > 0) & (df["RPM"] > 0)].copy()
df_bingham["log_ROP"] = np.log(df_bingham["ROP"])

# Fit Log-MLR
X_log = sm.add_constant(df_bingham[["WOB", "RPM"]])
log_model = sm.OLS(df_bingham["log_ROP"], X_log).fit()

# Bingham model
def bingham_model(X, K, a):
    wob, rpm = X
    Db = 444.5  # bit diameter in mm
    return K * (wob / Db)**a * rpm

# Fit Bingham via MLE
X_data = np.vstack((df_bingham["WOB"].values, df_bingham["RPM"].values))
params_opt, _ = curve_fit(bingham_model, X_data, df_bingham["ROP"].values, p0=(1.0, 1.0), maxfev=20000)
K_hat, a_hat = params_opt
print(f"K = {K_hat:.6f}, a = {a_hat:.6f}")


## RMSE Comparison


In [ ]:

rop_bingham_pred = bingham_model(
    np.vstack((wob, rpm)),
    K_hat,
    a_hat
)

rmse_bingham = np.sqrt(np.mean((rop_obs - rop_bingham_pred)**2))
print("RMSE (Bingham Model) =", rmse_bingham)

df_compare = df_bingham.copy()

X_comp = df_compare[["WOB", "RPM"]]
X_comp = sm.add_constant(X_comp)

log_rop_pred = log_model.predict(X_comp)

rop_logmlr_pred = np.exp(log_rop_pred)

rmse_logmlr = np.sqrt(np.mean((df_compare["ROP"] - rop_logmlr_pred)**2))

print("RMSE (Log-MLR Model) =", rmse_logmlr)


## Model Comparison Plot — ROP vs RPM


In [ ]:
# Use a subsample for scatter so the plot is readable
df_sample = df_bingham.sample(n=min(3000, len(df_bingham)), random_state=42)

# Fix RPM at the median of this well
median_rpm = df_bingham["RPM"].median()
print("Median RPM used for curves:", median_rpm)


In [ ]:
median_wob = df_bingham["WOB"].median()
print("Median WOB used for curves:", median_wob)

rpm_grid = np.linspace(df_bingham["RPM"].min(), df_bingham["RPM"].max(), 200)

# ----- Log-MLR curve (WOB fixed) -----
X_grid_log_rpm = pd.DataFrame({
    "const": 1.0,
    "WOB": median_wob,
    "RPM": rpm_grid
})
log_rop_grid_rpm = log_model.predict(X_grid_log_rpm)
rop_logmlr_grid_rpm = np.exp(log_rop_grid_rpm)

# ----- Bingham curve (WOB fixed) -----
X_grid_bing_rpm = np.vstack((np.full_like(rpm_grid, median_wob), rpm_grid))
rop_bingham_grid_rpm = bingham_model(X_grid_bing_rpm, K_hat, a_hat)

plt.figure(figsize=(7, 5))
plt.scatter(df_sample["RPM"], df_sample["ROP"], alpha=0.2, s=8, label="Observed")
plt.plot(rpm_grid, rop_logmlr_grid_rpm, linewidth=2, label="Log-MLR (WOB fixed)")
plt.plot(rpm_grid, rop_bingham_grid_rpm, linewidth=2, linestyle="--", label="Bingham (WOB fixed)")

plt.xlabel("RPM")
plt.ylabel("ROP (m/h)")
plt.title(f"ROP vs RPM (WOB fixed at {median_wob:.1f})")
plt.legend()
plt.tight_layout()
plt.show()

